### Setup

In [1]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd

from common.utils import DataPreprocessor, FeatureEngineer, set_seed
from common.exp_data_utils import ExperimentDataPreprocessor
from common.eval import Evaluator

MOVIELENS_DATA_DIR = "../datasets/hetrec2011-movielens-2k-v2/user_ratedmovies.dat"
RANDOM_SEED = 42

# Initialize data processors
set_seed(RANDOM_SEED)
data_preprocessor = DataPreprocessor()
feature_engineer = FeatureEngineer()
experiment_data_preprocessor = ExperimentDataPreprocessor()
evaluator = Evaluator()


/Users/jefferybai/Desktop/Master/BILAB/Master Thesis/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Seed set to 42
Seed set to 42


random seed set to 42
numpy seed set to 42
torch seed set to 42
lightning seed set to 42
torch set to use deterministic algorithms


### Load and Process DataFrame

In [2]:
interaction_df = data_preprocessor.load_and_process_df(
    file_dir=MOVIELENS_DATA_DIR,
    year_range=(2006, 2008),
    rating_threshold=4.0, # replica 3.5 rating threshold
)
interaction_df.head()

Data count: 855598
Data count after filtering by year (2006, 2008): 480608
Num of distinct users: 2103
Num of distinct items: 9519
done!
------------------------------
Filtering by min user/item interactions (10/0):
Data count before: 480608
Data count after: 480448
done!
------------------------------
==== Final Data Info: ====
Data Year Range: (2006, 2008)
Rating Threshold: 4.0
Num of interactions: 480448
Num of distinct users: 2064
Num of distinct items: 9519


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1


### Join Side Information

In [3]:
interaction_info_df = data_preprocessor.join_item_features(
    df=interaction_df, actor_k=5, threshold=5
)
interaction_info_df.head()

extracting item features...
merging features...
interaction data count before merging: 480448
interaction data count after merging: 478404
done!


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label,actorID,country,directorID,directorName,genre
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0,"[jack_lemmon, walter_matthau, annmargret, burg...",USA,donald_petrie,Donald Petrie,"[Comedy, Romance, [PAD], [PAD], [PAD], [PAD], ..."
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1,"[[RARE], [RARE], [RARE], [RARE], [RARE]]",USA,[RARE],Siddharth Randeria,"[Sci-Fi, Thriller, [PAD], [PAD], [PAD], [PAD],..."
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1,"[mel_gibson, sophie_marceau, patrick_mcgoohan,...",USA,[RARE],Mel Gibson,"[Action, Drama, War, [PAD], [PAD], [PAD], [PAD..."
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0,"[[RARE], laura_linney, ernie_hudson_jr, tim_cu...",USA,frank_marshall,Frank Marshall,"[Action, Adventure, Mystery, Sci-Fi, [PAD], [P..."
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1,"[antonio_banderas, salma_hayek, 1142520-joaqui...",USA,robert_rodriguez,Robert Rodriguez,"[Action, Romance, Thriller, [PAD], [PAD], [PAD..."


### Prepare Train/Valid/Test Set

In [4]:
# TODO: determine which method to use for splitting
# 1. Split by year
# 2. Stratified split by user, timestamp

train_df, valid_df, test_df = experiment_data_preprocessor.stratified_time_split(
    interaction_info_df,
    time_col="timestamp",
    train_ratio=0.64, # replica 64% for training
    val_ratio=0.16, # replica 16% for validation
    test_ratio=0.20, # replica 20% for testing
)

TRAIN_NUM_USERS = len(train_df["userID"].unique())
TRAIN_NUM_ITEMS = len(train_df["movieID"].unique())


Splitting data into train/valid/test by time period with ratio=(0.64 : 0.16 : 0.2):
train: 305206 (63.8%)
valid: 75578 (15.8%)
test: 97620 (20.41%)
------------------------------ 

Check target label distribution after splitting (%):
train label
0    0.547669
1    0.452331
Name: proportion, dtype: float64
valid label
0    0.606737
1    0.393263
Name: proportion, dtype: float64
test label
0    0.590914
1    0.409086
Name: proportion, dtype: float64


### Re-index User/Item ID & Encode Categorical Features

In [5]:
print("Train: fit_transform")
encoded_train_df = feature_engineer.fit_transform(train_df)
print("---"*10)
print("Valid: transform")
encoded_valid_df = feature_engineer.transform(valid_df)
print("---"*10)
print("Test: transform")
encoded_test_df = feature_engineer.transform(test_df)
print("---"*10)

Train: fit_transform
Re-index mapping dumped into ...
user: ../datasets/userid_mapping.csv
item: ../datasets/itemid_mapping.csv
Fitted: user/item mapping
Fitted: vocab2idx for actorID
Fitted: vocab2idx for country
Fitted: vocab2idx for directorID
Fitted: vocab2idx for genre
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Valid: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Test: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------


In [6]:
# # NOTE: can check the encoding vocab idx content from the feature engineer
# oov_idx = feature_engineer.vocab2idx["movieID"]["[OOV]"]
# len(test_df[test_df["movieID"] == oov_idx])

### Prepare Additional Data for Train/Inference

#### Build bi-partite graph for training

In [7]:
# NOTE: At training, we use interaction graph from train_df for train and validation
train_graph = experiment_data_preprocessor.create_interaction_graph(encoded_train_df)

# NOTE: At inference, we can use graph of (train_df + valid_df)
# train_valid_graph = utils.create_interaction_graph(pd.concat([train_df, valid_df], axis=0))

Creating interaction graph...
Drop negative samples
  Num of all interactions: 305206
  Num of positive interactions: 138054 

Building edges...
Building labels...
Interaction Graph: Data(edge_index=[2, 276107], edge_label=[138054])
Edge Index: tensor([[    0,     0,     0,  ..., 10448, 10452, 10453],
        [ 2089,  2200,  2315,  ...,  1760,  1645,  1760]])


#### Evaluate User Diversity Preference Scale

In [8]:
user_dps_df = evaluator.eval_user_diversity_preference_scale(encoded_train_df, feature_engineer.vocab2idx, normalized=True, rescale=True)
user_dps_df.head(1)


Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:06<00:00, 306.42it/s]


,userID,actorID_wvec,actorID_dps,country_wvec,country_dps,directorID_wvec,directorID_dps,genre_wvec,genre_dps
0,0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.340786,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.177372,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.457422,"[74.5, 34.0, 8.0, 5.0, 35.5, 45.5, 0.0, 37.0, ...",0.724296


#### Prepare Item Multihot Vec on Each Dimension

In [9]:
item_vec_df = evaluator.get_item_feature_multihot_vec(encoded_train_df, feature_engineer.vocab2idx)
item_vec_df.head(1)

,movieID,actorID_vec,country_vec,directorID_vec,genre_vec
0,1556,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


#### Prepare train/valid triplet data

In [10]:
train_triplet_df = experiment_data_preprocessor.prepare_triplet_df(encoded_train_df, k_negative_samples=5)
train_triplet_with_dps_df = train_triplet_df.merge(user_dps_df, on="userID", how="left")
train_triplet_with_dps_df = train_triplet_with_dps_df.merge(item_vec_df, left_on="pos_item_id", right_on="movieID", how="left")
train_triplet_with_dps_df.head(1)

# # TODO: dataset exp (failed to improve performance)
# finetune_df = experiment_data_preprocessor.prepare_finetune_df(df=encoded_train_df)
# finetune_with_dps_df = finetune_df.merge(user_dps_df, on="userID", how="left")
# finetune_with_dps_df = finetune_with_dps_df.merge(item_vec_df, on="movieID", how="left")
# finetune_with_dps_df.head(1)

validation_pool_df = experiment_data_preprocessor.prepare_prediction_df(encoded_valid_df, encoded_train_df, K=1000)

Original data count (positive samples): 138054
Num of triplets: 138054(pos samples) * 5(negative sampled items) = 690270
Prediction DataFrame:
User Pool: 2064
Item Pool: 8397, negative sampled to 1000 items for each user
Num of interactions: 2064(users) * 1000(items) = 2064000


### Prepare DataLoader

In [ ]:
# NOTE: ensure reproducibility of DataLoader
import torch
from common.utils import seed_worker
g = torch.Generator()
g.manual_seed(RANDOM_SEED)

# TODO: determine which Dataset to use
from torch.utils.data import DataLoader
from common.datasets import TripletDataset, UserItemPairDataset, UserPosItemSampler, get_user_triplet_mapping

BATCH_SIZE = 1024

# train_dataset = UserItemPairDataset(finetune_with_dps_df) # TODO: dataset exp (failed to improve performance)
train_dataset = TripletDataset(train_triplet_with_dps_df)
# valid_dataset = TripletDataset(valid_triplet_with_dps_df) # TODO: early stopping on val loss
valid_dataset = UserItemPairDataset(validation_pool_df)
print("train data count:", len(train_dataset))
print("valid data count:", len(valid_dataset))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, worker_init_fn=seed_worker, generator=g, num_workers=4)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)


### Load Embeddings

In [ ]:
model_name = "ngcf" # (mt)ngcf_v2_k5, lightgcn_k5
embeddings_dir = "embeddings/ngcf/"
user_emb = torch.load(f"{embeddings_dir}user_emb.pt") # {model_name}_
item_emb = torch.load(f"{embeddings_dir}item_emb.pt") # {model_name}_

print(user_emb.size())
print(item_emb.size())


torch.Size([2065, 256])
torch.Size([8397, 256])


### Configure Model (LightningModule)

In [ ]:
from lightning_models.finetune import FTDPRec

LR = 1e-3
EPOCHS = 1
L2_REG_WEIGHT = 1e-3

DPS_WEIGHTS = {
    "actor_dps": 0.25,
    "country_dps": 0.25,
    "director_dps": 0.25,
    "genre_dps": 0.25,
}

DPR_WEIGHTS = {
    "actor_dpr": 0.25,
    "country_dpr": 0.25,
    "director_dpr": 0.25,
    "genre_dpr": 0.25,
}

DPM_WEIGHTS = {
    "actor_pd": 0.25,
    "country_pd": 0.25,
    "director_pd": 0.25,
    "genre_pd": 0.25,
}

MT_WEIGHTS = {
    "dps_loss": 1.0,
    "dpr_loss": 1.0,
    "dpm_loss": 1.0,
}

REL_EMB_DIM = None  # Dimension of relation embeddings, can be adjusted based on model complexity
STRATEGY = "delta"  # "direct", "delta", "lora"
LORA_R = 2  # Rank for LoRA, can be adjusted based on model complexity
RESCALE_METHOD = "gradnorm"  # None, "log", "ema", "gradnorm"

model = FTDPRec(
    user_emb_tensor=user_emb,
    item_emb_tensor=item_emb,
    strategy=STRATEGY,
    lora_r=LORA_R,
    rel_dim=REL_EMB_DIM,
    lr=LR,
    l2_reg_weight=L2_REG_WEIGHT,
    dps_weights=DPS_WEIGHTS,
    dpr_weights=DPR_WEIGHTS,
    dpm_weights=DPM_WEIGHTS,
    mt_weights=MT_WEIGHTS,
    rescale_method=RESCALE_METHOD,
)


Seed set to 42


### Configure Trainer and Experiment

In [ ]:
from common._mlflow import get_mlflow_logger, get_callbacks

EXPERIMENT_NAME = "finetune-exp"
VERSION = "ngcf-test"
RUN_NAME = "gradnorm"
PATIENCE = 5
mlflow_logger = get_mlflow_logger(experiment_name=EXPERIMENT_NAME, run_name=RUN_NAME, tags={"version": VERSION})
trainer_callbacks = get_callbacks(
    exp_name=EXPERIMENT_NAME,
    version_name=VERSION,
    run_name=RUN_NAME,
    patience=PATIENCE,
    monitor_metric="val_ndcg20",
    monitor_mode="max",
    hyper_param_str=f"base-model-{model_name}-lr={LR}",
)

In [ ]:
from pytorch_lightning import Trainer

trainer = Trainer(
    max_epochs=EPOCHS,
    logger=mlflow_logger,
    log_every_n_steps=50,
    callbacks=trainer_callbacks,
    accelerator='cpu',  # or 'auto', 'gpu'
    # devices=[1], # if gpu is available
)


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Users/jefferybai/Desktop/Master/BILAB/Master Thesis/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.


### Train Model

In [ ]:
# Start training
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=valid_loader)


/Users/jefferybai/Desktop/Master/BILAB/Master Thesis/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /Users/jefferybai/Desktop/Master/BILAB/Master Thesis/DPRecSys/experiments/test_checkpoints/finetune-exp exists and is not empty.

  | Name                | Type              | Params | Mode 
------------------------------------------------------------------
0 | embedding_model     | EmbeddingWrapper  | 5.4 M  | train
1 | l2_regularizer      | L2Loss            | 0      | train
2 | loss_scaling_module | LogScaleLoss      | 0      | train
3 | ema_normalizer      | EMALossNormalizer | 0      | train
4 | dps_module          | DPSPredictor      | 1.0 K  | train
5 | dps_loss_fn         | DPSLoss           | 0      | train
6 | dpr_module          | DPRegularizer     | 262 K  | train
7 | dpr_loss_fn         | DPRLoss           | 0      | train
8 | dpm_module          | DPMatcher         | 0      | train
9 | dpm_loss_fn       

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/Users/jefferybai/Desktop/Master/BILAB/Master Thesis/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/Users/jefferybai/Desktop/Master/BILAB/Master Thesis/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:420: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.


Training: |          | 0/? [00:00<?, ?it/s]

/Users/jefferybai/Desktop/Master/BILAB/Master Thesis/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/Users/jefferybai/Desktop/Master/BILAB/Master Thesis/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/Users/jefferybai/Desktop/Master/BILAB/Master Thesis/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/Users/jefferybai/Desktop/Master/BILAB/Master Thesis/DPRecSys/.venv/lib/pyth

🏃 View run gradnorm at: http://140.112.106.216:3683/#/experiments/17/runs/23ab9b001a6448c0b350527a5bdc68e6
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/17


RuntimeError: One of the differentiated Tensors does not require grad

In [ ]:
pri()

NameError: name 'pri' is not defined

### Inference

#### Prepare prediction pool for inference/testing

In [ ]:
# NOTE: Prepare prediction pool to evaluate the model
seen_df = pd.concat([encoded_train_df, encoded_valid_df], ignore_index=True)
prediction_pool_df = experiment_data_preprocessor.prepare_prediction_df(encoded_test_df, seen_df, K=-1)
prediction_pool_df.tail()

Prediction DataFrame:
User Pool: 2064
Item Pool: 8397, negative sampled to -1 items for each user
Num of interactions: 2064(users) * -1(items) = 16952432


,userID,movieID,label,actorID_idx,country_idx,directorID_idx,genre_idx
16952427,2063,8392,0,"[2214, 1834, 1886, 2036, 1214]",37,1,"[6, 0, 0, 0, 0, 0, 0, 0]"
16952428,2063,8393,0,"[1070, 110, 1, 1, 1]",12,308,"[9, 0, 0, 0, 0, 0, 0, 0]"
16952429,2063,8394,0,"[1782, 1, 1, 1, 0]",36,365,"[4, 6, 0, 0, 0, 0, 0, 0]"
16952430,2063,8395,0,"[1, 1733, 935, 24, 1]",37,443,"[9, 17, 18, 0, 0, 0, 0, 0]"
16952431,2063,8396,0,"[1, 1, 0, 0, 0]",17,1,"[9, 0, 0, 0, 0, 0, 0, 0]"


In [ ]:
# NOTE: Use UserItemPairDataset for prediction pool
test_dataset = UserItemPairDataset(prediction_pool_df)
print("test data count:", len(test_dataset))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


test data count: 16952432


In [ ]:
# NOTE: the inference model MUST be the same as the training model
# best_model_experiment_name = "finetune-exp"
# best_model_checkpoint_path = "[ngcf]-all-task-base-model-ngcf_v2_k5-lr=0.001-best-checkpoint-epoch=00-val_ndcg20=0.37.ckpt"
# best_model_path = f"test_checkpoints/{best_model_experiment_name}/{best_model_checkpoint_path}"
best_model_path = trainer.checkpoint_callback.best_model_path

model = FTDPRec.load_from_checkpoint(checkpoint_path=best_model_path)
# start inference
trainer.test(model=model, dataloaders=test_loader)


Seed set to 42
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        test_ndcg10        │    0.19627468287944794    │
│        test_ndcg20        │    0.22327791154384613    │
│        test_ndcg5         │    0.15765395760536194    │
│     test_precision10      │    0.05886627733707428    │
│     test_precision20      │    0.05101744085550308    │
│      test_precision5      │    0.06308139860630035    │
│       test_recall10       │    0.03716747462749481    │
│       test_recall20       │    0.06299611926078796    │
│       test_recall5        │    0.02014526166021824    │
└───────────────────────────┴───────────────────────────┘

🏃 View run all-task at: http://140.112.106.216:3683/#/experiments/17/runs/afed5da0e632422da041a369ee491931
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/17


[{'test_ndcg5': 0.15765395760536194,
  'test_ndcg10': 0.19627468287944794,
  'test_ndcg20': 0.22327791154384613,
  'test_precision5': 0.06308139860630035,
  'test_precision10': 0.05886627733707428,
  'test_precision20': 0.05101744085550308,
  'test_recall5': 0.02014526166021824,
  'test_recall10': 0.03716747462749481,
  'test_recall20': 0.06299611926078796}]

In [ ]:
model.test_results["eval_score_df"].describe()

,user,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.157654,0.020145,0.063081,0.196275,0.037167,0.058866,0.223278,0.062996,0.051017
std,595.969798,0.303294,0.056298,0.129106,0.291815,0.080525,0.097913,0.270675,0.105477,0.075398
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,515.750000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1031.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1547.250000,0.000000,0.000000,0.000000,0.381622,0.045996,0.100000,0.386853,0.090909,0.100000
max,2063.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.700000,1.000000,1.000000,0.550000


In [ ]:
# Evaluate user diversity preference matching score (DPMS) at k
eval_df = evaluator.prepare_evaluation_data(model.test_results, feature_engineer.idx2vocab)

# Get user DPMS
user_dpms_df = evaluator.evaluate_dpms_at_k(
    eval_df=eval_df,
    feature_engineer=feature_engineer,
    ground_truth_dps_df=user_dps_df,
    k=10,
    actor_k=5,
    rare_threshold=5,
)

user_dpms_df.describe()

candidate item pool size: 10
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 20640
interaction data count after merging: 20640
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064


Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:04<00:00, 503.74it/s]

combined_df 2064
Calculating DPMS for each feature...


,userID,actorID_dpms,country_dpms,directorID_dpms,genre_dpms,avg_dpms
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.104595,0.978127,0.116930,0.803630,0.500821
std,595.969798,0.049663,0.040802,0.093731,0.112340,0.046782
min,0.000000,0.000000,0.287681,0.000000,0.168947,0.199860
25%,515.750000,0.069264,0.976597,0.045335,0.744448,0.472916
50%,1031.500000,0.104749,0.988066,0.105023,0.826363,0.503983
75%,1547.250000,0.139219,0.994088,0.176131,0.890809,0.531604
max,2063.000000,0.278360,1.000000,0.514811,0.976876,0.650119


In [ ]:
# ILS@10
ils_df = evaluator.evaluate_ils_at_k(eval_df, k=10)
ils_df.describe()

,user,ILS@10
count,2064.000000,2064.000000
mean,35564.367733,0.319797
std,20797.975208,0.035509
min,75.000000,0.114566
25%,17798.500000,0.303715
50%,35054.000000,0.321800
75%,53331.000000,0.338607
max,71534.000000,0.445476


In [ ]:
embedding_path = "embeddings/finetune/"
prefix = "ngcf"
if not os.path.exists(embedding_path):
    os.makedirs(embedding_path)

torch.save(model.user_emb.cpu(), f"{embedding_path}{prefix}_user_emb.pt")
torch.save(model.item_emb.cpu(), f"{embedding_path}{prefix}_item_emb.pt")

In [ ]:
import joblib
dir = "artifacts/finetune"
prefix = "finetune_ngcf_k_all"
if not os.path.exists(dir):
    os.makedirs(dir)

joblib.dump(eval_df, os.path.join(dir, f"{prefix}_eval_df.pkl"))
joblib.dump(user_dpms_df, os.path.join(dir, f"{prefix}_user_dpms_df.pkl"))
joblib.dump(user_dps_df, os.path.join(dir, f"user_dps_df.pkl"))
joblib.dump(feature_engineer, os.path.join(dir, f"feature_engineer.pkl"))

['artifacts/finetune/feature_engineer.pkl']